# 05 - Factor Robustness

The interview-prep notebook. Three robustness lenses on a chosen factor: its
worst months (the momentum-crash question), FF5 exposures of its long-short
return (is the alpha just known risk factors?), and the IC decay curve.

**Prerequisite:** `make data`. FF5 exposures additionally need an FF5 factor
file at `config.FF5_FILE` (Ken French data); the cell skips cleanly if absent.

In [1]:
from qer.data.loader import DataLoader
from qer.factors import get_factor
from qer.diagnostics.portfolios import factor_long_short, ic_decay
import pandas as pd

loader = DataLoader()
close = loader.close
start = close.index.max() - pd.DateOffset(years=10)
dates = close.index[(close.index >= start) & (close.index <= close.index[-22])]
dates = dates[dates >= close.index[273]]
fac = get_factor("momentum_12_1")

## Worst months

Aggregate the long-short daily return to monthly and show the worst - have the
explanation ready (e.g. momentum crashes in the sharp post-drawdown recoveries
of 2009, 2016, 2020, when beaten-down names lead).

In [2]:
ls = factor_long_short(loader, fac, n_buckets=10, horizon=21, dates=dates)
monthly = ls.resample("ME").sum()
monthly.sort_values().head(8)

2020-05-31   -3.785941
2020-10-31   -3.263931
2020-11-30   -2.343646
2021-02-28   -2.328739
2022-12-31   -1.833874
2023-11-30   -1.800664
2019-08-31   -1.749128
2023-01-31   -1.697633
Name: long_short, dtype: float64

## IC decay curve

Mean IC as the forward horizon lengthens - where the signal lives and where it
fades.

In [3]:
ic_decay(loader, fac, horizons=(1,5,10,21,42,63), dates=dates)

1     0.013494
5     0.006953
10   -0.000677
21   -0.008751
42   -0.010456
63   -0.010706
Name: mean_ic, dtype: float64

## FF5 exposures

A *time-series* regression of the long-short return on the Fama-French 5 factors
(distinct from the cross-sectional IC). The intercept is the FF5-adjusted alpha;
the question is whether it survives the known risk factors.

The test is **H0: alpha = 0**. A factor earns its keep only if we can *reject* that:
a **positive** FF5-adjusted alpha that is statistically non-zero (NW |t| roughly >= 2)
once the known risk factors are stripped out. A positive alpha with |t| < 2 is "not
distinguishable from zero", not a finding. Caveat: this is one factor among several
tested, so the honest significance bar is the cross-factor correction in notebook 04
(BH / deflated Sharpe), not this single t-stat in isolation.

In [4]:
from qer.config import FF5_FILE
from qer.diagnostics.exposures import ff5_exposures
if FF5_FILE.exists():
    ff5 = pd.read_parquet(FF5_FILE)
    res = ff5_exposures(ls, ff5)
    a, t = res["alpha"], res["t_stats_nw"]["alpha"]   # NW-labelled alias
    verdict = "reject H0 (non-zero alpha)" if abs(t) >= 2 else "cannot reject H0"
    print(f"alpha: {a:+.6f} | alpha t ({res['t_stat_kind']}): {t:+.2f} -> {verdict}")
    print("betas:", {k: round(v, 3) for k, v in res["betas"].items()})
else:
    print(f"FF5 file not found at {FF5_FILE}; download Ken French FF5 to enable.")

alpha: +0.009971 | alpha t (newey_west_hac(lags=21)): +1.94 -> cannot reject H0
betas: {'mkt_rf': np.float64(-0.345), 'smb': np.float64(-0.033), 'hml': np.float64(0.382), 'rmw': np.float64(-0.562), 'cma': np.float64(-0.221)}


## Takeaways

A factor worth keeping has a worst-month story you can explain, an IC that
decays gracefully rather than reversing, and an FF5-adjusted alpha that does not
vanish once the known factors are removed.